In [ ]:
!pip install pysrt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pysrt: filename=pysrt-1.1.2-py3-none-any.whl size=13443 sha256=dc474ea31b4e5bd40cf503410b819ce374c0feaebba48e4e73c127dacbe70f6d
  Stored in directory: /root/.cache/pip/wheels/6a/36/54/2aa8dc961885dfa7b0ebd45a57505f25039d79b4ea0fd9f29d
Successfully built pysrt


In [ ]:
import requests
import time
import zipfile
import io
import pysrt
import pandas as pd

API_KEY = "OLe6fdZ1yXNx5g2Nc4Ewjtx6V1Tl0ALf"
USER_AGENT = "SubtitleAligner v1.0"

HEADERS = {
    "Api-Key": API_KEY,
    "User-Agent": USER_AGENT,
    "Content-Type": "application/json"
}


# ---------------------------
# 1. SEARCH SUBTITLES
# ---------------------------
def search_subtitles(query, language):
    url = "https://api.opensubtitles.com/api/v1/subtitles"

    params = {
        "query": query,
        "languages": language
    }

    res = requests.get(url, headers=HEADERS, params=params)
    data = res.json()

    if "data" not in data:
        raise Exception(f"API error: {data}")

    return data["data"]


# ---------------------------
# 2. DOWNLOAD SUBTITLE FILE
# ---------------------------
def download_subtitle(file_id):
    url = "https://api.opensubtitles.com/api/v1/download"

    res = requests.post(url, headers=HEADERS, json={"file_id": file_id})
    data = res.json()

    if "link" not in data:
        raise Exception(f"Download error: {data}")

    download_url = data["link"]

    file_res = requests.get(download_url)

    return file_res.content


# ---------------------------
# 3. SAVE SRT FILE
# ---------------------------
def save_srt(content, filename):
    # Sometimes it's zipped
    try:
        z = zipfile.ZipFile(io.BytesIO(content))
        for name in z.namelist():
            if name.endswith(".srt"):
                with open(filename, "wb") as f:
                    f.write(z.read(name))
                return
    except:
        pass

    # Otherwise raw file
    with open(filename, "wb") as f:
        f.write(content)


# ---------------------------
# 4. GET FIRST MATCHING SUBTITLE
# ---------------------------
def get_subtitle_file_id(results):
    for item in results:
        files = item["attributes"]["files"]
        if files:
            return files[0]["file_id"]
    raise Exception("No valid subtitle file found")


# ---------------------------
# 5. ALIGN SUBTITLES
# ---------------------------
def align_subtitles(subs_de, subs_en, threshold_ms=2000):
    aligned = []
    i, j = 0, 0

    while i < len(subs_de) and j < len(subs_en):
        de = subs_de[i]
        en = subs_en[j]

        diff = abs(de.start.ordinal - en.start.ordinal)

        if diff < threshold_ms:
            aligned.append({
                "start": str(de.start),
                "end": str(de.end),
                "german": de.text.replace("\n", " "),
                "english": en.text.replace("\n", " ")
            })
            i += 1
            j += 1
        elif de.start.ordinal < en.start.ordinal:
            i += 1
        else:
            j += 1

    return aligned


# ---------------------------
# 6. MAIN PIPELINE
# ---------------------------
def run_pipeline(movie_name):
    print(f"Searching subtitles for: {movie_name}")

    # Search German + English
    de_results = search_subtitles(movie_name, "de")
    en_results = search_subtitles(movie_name, "en")

    # Get file IDs
    de_file_id = get_subtitle_file_id(de_results)
    en_file_id = get_subtitle_file_id(en_results)

    print("Downloading subtitles...")

    # Download files
    de_content = download_subtitle(de_file_id)
    time.sleep(1)  # avoid rate limits
    en_content = download_subtitle(en_file_id)

    # Save locally
    save_srt(de_content, "movie.de.srt")
    save_srt(en_content, "movie.en.srt")

    print("Parsing subtitles...")

    subs_de = pysrt.open("movie.de.srt")
    subs_en = pysrt.open("movie.en.srt")

    print("Aligning subtitles...")

    aligned = align_subtitles(subs_de, subs_en)

    print(f"Aligned {len(aligned)} subtitle pairs")

    # Export to Excel
    df = pd.DataFrame(aligned)
    df.to_excel("aligned_subtitles.xlsx", index=False)

    print("Saved to aligned_subtitles.xlsx")


if __name__ == "__main__":
    movie = input("Enter movie name: ")
    run_pipeline(movie)

Enter movie name: Das Leben der Anderen
Searching subtitles for: Das Leben der Anderen
Parsing subtitles...
Aligning subtitles...
Aligned 737 subtitle pairs
Saved to aligned_subtitles.xlsx ✅
